<a href="https://colab.research.google.com/github/awfajri/artificial-intelligence/blob/main/Tugas_Pertemuan_3_Auf_Fajri_Ramadhani.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tugas Pertemuan 3 - Auf Fajri Ramadhani




Kelas : 4F

NPM : 2410631170059



---



#**Soal**

1.   Jelaskan setiap tahapan pada kode diatas!
2.   Apa output yang didapat dari kode diatas? jelaskan!




#**Jawab**

#1. Mengimpor Library yang Dibutuhkan
Tahapan pertama adalah memuat modul bawaan Python yang akan digunakan. csv digunakan untuk membaca file dataset, random untuk membagi data secara acak, dan math untuk melakukan operasi matematika seperti akar kuadrat dan eksponensial dalam rumus Gaussian.

In [ ]:
from csv import reader
from random import seed
from random import randrange
from math import sqrt
from math import exp
from math import pi

#2. Membaca Dataset (load_csv)
Fungsi ini digunakan untuk membuka dan membaca file iris.csv baris demi baris. Data yang dibaca kemudian dimasukkan ke dalam sebuah list. Baris yang kosong akan dilewati agar tidak menyebabkan error

In [ ]:
def load_csv(filename):
    dataset = list()
    with open(filename, 'r') as file:
        csv_reader = reader(file)
        for row in csv_reader:
            if not row:
                continue
            dataset.append(row)
    return dataset

#3. Konversi Tipe Data Teks ke Desimal (str_column_to_float)
Karena data yang dibaca dari file CSV secara default berformat teks (string), fungsi ini bertugas mengubah nilai ukuran kelopak bunga menjadi angka desimal (float) agar bisa dihitung secara matematis .

In [ ]:
def str_column_to_float(dataset, column):
    for row in dataset:
        row[column] = float(row[column].strip())

#4. Konversi Nama Kelas ke Angka (str_column_to_int)
Fungsi ini mengubah label kelas atau jenis bunga (misal: Iris-setosa, Iris-virginica) yang ada di kolom terakhir menjadi angka indeks (0, 1, 2) menggunakan bantuan dictionary (kamus) .

In [ ]:
def str_column_to_int(dataset, column):
    class_values = [row[column] for row in dataset]
    unique = set(class_values)
    lookup = dict()
    for i, value in enumerate(unique):
        lookup[value] = i
    for row in dataset:
        row[column] = lookup[row[column]]
    return lookup

#5. Fungsi Evaluasi dan Cross Validation
Kumpulan fungsi ini bertugas membagi dataset menjadi beberapa lipatan (folds) secara acak untuk keperluan pengujian model. Tujuannya adalah untuk menghitung persentase seberapa akurat algoritma ini dalam menebak data uji secara keseluruhan .

In [ ]:
# Split a dataset into k folds
def cross_validation_split(dataset, n_folds):
    dataset_split = list()
    dataset_copy = list(dataset)
    fold_size = int(len(dataset) / n_folds)
    for i in range(n_folds):
        fold = list()
        while len(fold) < fold_size:
            index = randrange(len(dataset_copy))
            fold.append(dataset_copy.pop(index))
        dataset_split.append(fold)
    return dataset_split

# Calculate accuracy percentage
def accuracy_metric(actual, predicted):
    correct = 0
    for i in range(len(actual)):
        if actual[i] == predicted[i]:
            correct += 1
    return correct / float(len(actual)) * 100.0

# Evaluate an algorithm using a cross validation split
def evaluate_algorithm(dataset, algorithm, n_folds, *args):
    folds = cross_validation_split(dataset, n_folds)
    scores = list()
    for fold in folds:
        train_set = list(folds)
        train_set.remove(fold)
        train_set = sum(train_set, [])
        test_set = list()
        for row in fold:
            row_copy = list(row)
            test_set.append(row_copy)
            row_copy[-1] = None
        predicted = algorithm(train_set, test_set, *args)
        actual = [row[-1] for row in fold]
        accuracy = accuracy_metric(actual, predicted)
        scores.append(accuracy)
    return scores

#6. Pemisahan Data Berdasarkan Kelas (separate_by_class)
Fungsi ini mengelompokkan dan memisahkan seluruh baris data latih berdasarkan label kelasnya (jenis bunganya) ke dalam sebuah dictionary .

In [ ]:
def separate_by_class(dataset):
    separated = dict()
    for i in range(len(dataset)):
        vector = dataset[i]
        class_value = vector[-1]
        if (class_value not in separated):
            separated[class_value] = list()
        separated[class_value].append(vector)
    return separated

#7. Perhitungan Mean dan Standar Deviasi
Dua fungsi ini merupakan dasar perhitungan statistik. mean mencari nilai rata-rata dari sekumpulan angka, dan stdev mencari nilai standar deviasi (sebaran data)

In [ ]:
def mean(numbers):
    return sum(numbers)/float(len(numbers))

def stdev(numbers):
    avg = mean(numbers)
    variance = sum([(x-avg)**2 for x in numbers]) / float(len(numbers)-1)
    return sqrt(variance)

#8. Meringkas Data (summarize_dataset, summarize_by_class)
Fungsi ini merangkum setiap kolom atribut dengan menghitung nilai mean, standar deviasi, dan jumlah datanya . Rangkuman ini kemudian diterapkan secara spesifik untuk masing-masing kelas bunga yang sudah dipisah sebelumnya .

In [ ]:
def summarize_dataset(dataset):
    summaries = [(mean(column), stdev(column), len(column)) for column in zip(*dataset)]
    del(summaries[-1])
    return summaries

def summarize_by_class(dataset):
    separated = separate_by_class(dataset)
    summaries = dict()
    for class_value, rows in separated.items():
        summaries[class_value] = summarize_dataset(rows)
    return summaries

#9. Perhitungan Probabilitas Gaussian
Ini adalah inti dari metode Naive Bayes. Fungsi pertama menerapkan rumus Distribusi Probabilitas Gaussian untuk mencari nilai peluang . Fungsi kedua mengalikan nilai-nilai peluang dari setiap atribut secara independen untuk mendapatkan total probabilitas sebuah baris data baru terhadap masing-masing kelas bunga

In [ ]:
def calculate_probability(x, mean, stdev):
    exponent = exp(-((x-mean)**2 / (2 * stdev**2)))
    return (1 / (sqrt(2 * pi) * stdev)) * exponent

def calculate_class_probabilities(summaries, row):
    total_rows = sum([summaries[label][0][2] for label in summaries])
    probabilities = dict()
    for class_value, class_summaries in summaries.items():
        probabilities[class_value] = summaries[class_value][0][2]/float(total_rows)
        for i in range(len(class_summaries)):
            mean, stdev, _ = class_summaries[i]
            probabilities[class_value] *= calculate_probability(row[i], mean, stdev)
    return probabilities

#10. Fungsi Prediksi (predict)
Fungsi ini membandingkan hasil perhitungan total probabilitas dari tiap kelas. Label kelas yang memiliki nilai probabilitas paling tinggi akan ditetapkan sebagai hasil tebakan final (best_label) .

In [ ]:
def predict(summaries, row):
    probabilities = calculate_class_probabilities(summaries, row)
    best_label, best_prob = None, -1
    for class_value, probability in probabilities.items():
        if best_label is None or probability > best_prob:
            best_prob = probability
            best_label = class_value
    return best_label

#11. Eksekusi Program Utama
Tahapan terakhir ini adalah tempat program benar-benar berjalan. Program memuat dataset iris.csv, menghapus baris judul (header), memformat data, melakukan peringkasan statistik (melatih model), lalu memberikan sebuah data kelopak bunga baru [5.7, 2.9, 4.2, 1.3] untuk ditebak oleh sistem .

In [ ]:
filename = 'iris.csv'
dataset = load_csv(filename)
dataset.pop(0)

for i in range(len(dataset[0])-1):
    str_column_to_float(dataset, i)
str_column_to_int(dataset, len(dataset[0])-1)

model = summarize_by_class(dataset)
row = [5.7, 2.9, 4.2, 1.3]

label = predict(model, row)
print('Data=%s, Predicted: %s' % (row, label))

Data=[5.7, 2.9, 4.2, 1.3], Predicted: 0


#Penjelasan Output:
Output tersebut menunjukkan bahwa sistem AI telah berhasil mengenali data uji. Bagian Data=[...] menampilkan ukuran bunga acak yang diinputkan oleh pengguna. Kemudian, sistem merespons dengan Predicted: 0, yang berarti setelah menghitung dan membandingkan probabilitas Gaussian, algoritma Naive Bayes menyimpulkan bahwa bunga tersebut paling cocok masuk ke dalam karakteristik kelas berindeks 0.